In [30]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from tabulate import tabulate

plt.style.use("../auri.mplstyle")
pd.options.display.unicode.east_asian_width = True

In [31]:
parquet_path = Path("../data/processed/주건축물_검증규칙.parquet")
df_orig = pd.read_parquet(parquet_path)
print(f"Loaded shape: {df_orig.shape}")
with pd.option_context("display.max_columns", None):
    print(tabulate(df_orig.head(30), headers="keys", tablefmt="psql"))

Loaded shape: (7364967, 38)
+----+----------------------+---------------------------+--------+-------------------------------------+---------------+---------------+----------------+-------------------+--------------------------------------+-----------------+-----------------+--------------+--------------+---------------------------+--------------+-----------+--------------+--------------+---------------+----------------------+----------------------+-------------------+-------------------+--------------------------------+-------------------+---------------------+----------+----------+-----------------+---------------+----------+-----------------+---------------+----------+----------+---------------+----------+----------+
|    |   관리_건축물대장_PK |   관리_상위_건축물대장_PK | 단독   | 대지_위치                           |   시군구_코드 |   법정동_코드 |   주_용도_코드 | 주_용도_코드_명   | 기타_용도                            |   대지_면적(㎡) |   건축_면적(㎡) |   건폐_율(%) |   연면적(㎡) |   용적_률_산정_연면적(㎡) |   용적_률(%) |   높이(m) |   지상_층_수 |   지

In [32]:
df_results = pd.read_parquet("../results/area_data_anomaly.parquet")
df_results = df_results.convert_dtypes()
print(f"Loaded shape: {df_results.shape}")
with pd.option_context("display.max_columns", None):
    print(tabulate(df_results.head(30), headers="keys", tablefmt="psql"))

Loaded shape: (7364967, 11)
+----+----------------------+-------------+---------------+----------------+----------+----------+----------+----------+----------+----------+----------+
|    |   관리_건축물대장_PK |   시도_코드 | 사용승인_년   |   주_용도_코드 | 기존14   | 기존16   | 기존19   | 기존20   | 신규01   | 신규02   | 이상값   |
|----+----------------------+-------------+---------------+----------------+----------+----------+----------+----------+----------+----------+----------|
|  0 |             10021950 |          11 | <NA>          |          01000 | <NA>     | <NA>     | <NA>     | <NA>     | <NA>     | <NA>     | <NA>     |
|  1 |             10021957 |          11 | 2000          |          02000 | False    | False    | False    | True     | False    | False    | False    |
|  2 |             10021958 |          11 | 1976          |          02000 | <NA>     | <NA>     | <NA>     | <NA>     | <NA>     | <NA>     | <NA>     |
|  3 |             10021973 |          11 | 1969          |          04000 | <NA>   

In [33]:
df_results.dtypes

관리_건축물대장_PK    string[python]
시도_코드             string[python]
사용승인_년           string[python]
주_용도_코드          string[python]
기존14                       boolean
기존16                       boolean
기존19                       boolean
기존20                       boolean
신규01                       boolean
신규02                       boolean
이상값                       boolean
dtype: object

In [34]:
df_results.head()

,관리_건축물대장_PK,시도_코드,사용승인_년,주_용도_코드,기존14,기존16,기존19,기존20,신규01,신규02,이상값
0,10021950,11,<NA>,01000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,10021957,11,2000,02000,False,False,False,True,False,False,False
2,10021958,11,1976,02000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,10021973,11,1969,04000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,10021974,11,1960,04000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


---

시도별, 연도별 표를 만들어보자.


In [35]:
df_year = pd.DataFrame(
    {
        "사용승인_년": [f"{year:04d}" for year in range(1950, 2025)],
    }
)
df_year

,사용승인_년
0,1950
1,1951
2,1952
3,1953
4,1954
...,...
70,2020
71,2021
72,2022
73,2023


In [36]:
df_usage = df_orig[["주_용도_코드", "주_용도_코드_명"]].drop_duplicates()
df_usage = df_usage.set_index("주_용도_코드").sort_index()
df_usage = df_usage.reindex(
    index=[f"{i:02d}000" for i in range(1, 34)]
)  # 01000, 02000, …
df_usage

,주_용도_코드_명
주_용도_코드,
01000,단독주택
02000,공동주택
03000,제1종근린생활시설
04000,제2종근린생활시설
05000,문화및집회시설
06000,종교시설
07000,판매시설
08000,운수시설
09000,의료시설


In [37]:
df_kcad = pd.read_csv(
    "../data/processed/code_kcad_sgg_2024.csv", dtype="string"
).sort_values("행정구역분류")
df_kcad["시도_코드"] = df_kcad["시군구코드"].str[:2]
df_sido = df_kcad[["시도_코드", "시도"]].drop_duplicates()
df_sido

,시도_코드,시도
0,11,서울특별시
25,26,부산광역시
41,27,대구광역시
50,28,인천광역시
60,29,광주광역시
65,30,대전광역시
70,31,울산광역시
75,36,세종특별자치시
76,41,경기도
127,51,강원특별자치도


In [38]:
df_results.shape

(7364967, 11)

In [39]:
df_results["기존16"].value_counts(dropna=False)

기존16
False    6647554
True      373505
<NA>      343908
Name: count, dtype: Int64

In [40]:
# mean()은 NA 제외하고 계산

print(df_results["기존16"].mean())
print(373505 / (7364967 - 343908))

0.05319781531532494
0.05319781531532494


In [41]:
# NA → False로 처리

print(df_results["기존16"].astype("boolean").fillna(False).mean())
print(373505 / (7364967))

0.05071373707444989
0.05071373707444989


In [42]:
# NA → False로 처리한 df 준비
# Convert columns to the best possible dtypes using dtypes supporting pd.NA
df_gb = df_results.iloc[:, 1:].copy().convert_dtypes()
df_gb.iloc[:, 3:] = df_gb.iloc[:, 3:].fillna(False)  # NA → False
df_gb

,시도_코드,사용승인_년,주_용도_코드,기존14,기존16,기존19,기존20,신규01,신규02,이상값
0,11,<NA>,01000,False,False,False,False,False,False,False
1,11,2000,02000,False,False,False,True,False,False,False
2,11,1976,02000,False,False,False,False,False,False,False
3,11,1969,04000,False,False,False,False,False,False,False
4,11,1960,04000,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
7364962,46,2014,01000,False,False,False,False,False,False,False
7364963,46,2014,01000,False,False,False,False,False,False,False
7364964,46,2014,04000,False,False,False,False,False,False,False
7364965,46,2014,21000,False,False,False,False,False,False,False


In [43]:
df_gb.dtypes

시도_코드       string[python]
사용승인_년     string[python]
주_용도_코드    string[python]
기존14                 boolean
기존16                 boolean
기존19                 boolean
기존20                 boolean
신규01                 boolean
신규02                 boolean
이상값                 boolean
dtype: object

In [44]:
gb = (
    df_gb.iloc[:, [0, *range(3, df_gb.shape[1])]]
    .groupby("시도_코드")
    .mean()
    .astype(float)
    .mul(100)
    .round(2)
)
df_ratio = df_sido.set_index("시도_코드").join(gb)
df_ratio

,시도,기존14,기존16,기존19,기존20,신규01,신규02,이상값
시도_코드,,,,,,,,
11,서울특별시,0.08,0.25,1.09,0.88,0.09,0.19,0.70
26,부산광역시,0.09,1.77,1.17,1.06,0.17,0.47,0.43
27,대구광역시,0.15,2.95,1.31,1.59,0.31,0.64,0.33
28,인천광역시,0.10,2.25,0.80,1.00,0.24,0.45,0.26
29,광주광역시,0.08,1.59,4.14,2.65,0.51,1.00,0.19
30,대전광역시,0.09,2.21,1.25,1.58,0.21,0.36,0.20
31,울산광역시,0.07,5.42,2.92,3.11,0.38,0.86,0.21
36,세종특별자치시,0.30,6.58,0.42,0.56,0.68,1.21,0.07
41,경기도,0.10,3.33,1.13,1.07,0.31,0.82,0.29


In [45]:
df_ratio.to_csv(
    "../results/area_data_anomaly_ratio_by_sido.csv", index=True, encoding="utf-8-sig"
)

In [26]:
df_orig["기존14"].value_counts(dropna=False)

기존14
False    5190727
None     2164467
True        9773
Name: count, dtype: int64

In [27]:
df_orig[df_orig["시군구_코드"].str.startswith("38")]

,관리_건축물대장_PK,관리_상위_건축물대장_PK,단독,대지_위치,시군구_코드,법정동_코드,주_용도_코드,주_용도_코드_명,기타_용도,대지_면적(㎡),...,건폐율_재계산,건폐율_차이,기존19,용적률_재계산,용적률_차이,기존20,신규01,연면적_상한,신규02,신규03
3405802,12311844,None,True,974-2번지,38022,38022,01000,단독주택,단독주택,NaN,...,NaN,NaN,None,NaN,NaN,None,False,NaN,None,False


---

연도


In [50]:
# Convert columns to the best possible dtypes using dtypes supporting pd.NA
df_gb = df_results.iloc[:, 1:].copy().convert_dtypes()
df_gb.iloc[:, 3:] = df_gb.iloc[:, 3:].fillna(False)  # NA → False

key_col = "사용승인_년"

gb = (
    df_gb.iloc[:, [1, *range(3, df_gb.shape[1])]]
    .groupby(key_col)
    .mean()
    .astype(float)
    .mul(100)
    .round(2)
)

df_ratio_year = df_year.set_index(key_col).join(gb).fillna("-")
with pd.option_context("display.max_rows", None):
    display(df_ratio_year)

,기존14,기존16,기존19,기존20,신규01,신규02,이상값
사용승인_년,,,,,,,
1950,0.29,0.44,0.58,0.50,0.19,0.19,0.29
1951,0.43,0.36,0.49,0.40,0.14,0.20,0.25
1952,0.28,0.33,0.58,0.52,0.16,0.12,0.37
1953,0.31,0.33,0.53,0.46,0.14,0.13,0.33
1954,0.24,0.30,0.45,0.40,0.14,0.15,0.33
1955,0.21,0.43,0.61,0.51,0.17,0.17,0.31
1956,0.22,0.44,0.62,0.57,0.17,0.16,0.29
1957,0.22,0.42,0.54,0.51,0.18,0.16,0.35
1958,0.23,0.38,0.76,0.66,0.18,0.16,0.37


In [51]:
df_ratio_year.to_csv(
    "../results/area_data_anomaly_ratio_by_year.csv", index=True, encoding="utf-8-sig"
)

---

용도


In [53]:
# Convert columns to the best possible dtypes using dtypes supporting pd.NA
df_gb = df_results.iloc[:, 1:].copy().convert_dtypes()
df_gb.iloc[:, 3:] = df_gb.iloc[:, 3:].fillna(False)  # NA → False

key_col = "주_용도_코드"

gb = (
    df_gb.iloc[:, [2, *range(3, df_gb.shape[1])]]
    .groupby(key_col)
    .mean()
    .astype(float)
    .mul(100)
    .round(2)
)

df_ratio_usage = df_usage.join(gb).fillna("-")
with pd.option_context("display.max_rows", None):
    display(df_ratio_usage)

,주_용도_코드_명,기존14,기존16,기존19,기존20,신규01,신규02,이상값
주_용도_코드,,,,,,,,
01000,단독주택,0.11,5.91,0.99,0.87,0.23,0.48,0.31
02000,공동주택,0.25,0.85,3.12,3.46,0.29,0.39,0.38
03000,제1종근린생활시설,0.24,4.27,1.08,1.08,0.29,0.70,0.33
04000,제2종근린생활시설,0.12,3.77,0.96,1.00,0.31,0.82,0.35
05000,문화및집회시설,0.12,7.99,0.85,0.75,0.30,0.94,0.30
06000,종교시설,0.13,5.02,1.38,1.48,0.52,1.32,0.31
07000,판매시설,0.34,3.90,1.11,1.21,0.26,0.94,0.42
08000,운수시설,0.15,13.34,1.22,0.87,0.44,1.10,0.29
09000,의료시설,0.17,1.25,0.73,1.00,0.30,0.90,0.36


In [54]:
df_ratio_usage.to_csv(
    "../results/area_data_anomaly_ratio_by_usage.csv", index=True, encoding="utf-8-sig"
)